# 📋 [OPS08_M05] 잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m05_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M05
- **미션 주제:** 잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 05. 잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"무작위 12개 사건(E01~E12)에 대해 6대 교차확인 규칙을 적용하여 실제 위협과 오경보를 분류하고 그 전술적 근거를 교차검증 매트릭스로 정리해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 실제 위협을 오경보로 놓친 미탐(Miss)이 0건인지 엄격히 검증하고, 판정 근거의 타당성을 확인합니다.

---

### 📑 실습자료 및 과업 지침

잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련**

가. 제공 자료 - 교차 확인 규칙(초안, 조별로 보완)

| **규칙** | **내용** |
| --- | --- |
| **R1 복창** | 지휘자는 받은 보고의 방위·수량을 복창하고 관측자가 '맞음/틀림'으로 답한다. |
| **R2 2차 관측** | 가능하면 다른 관측자에게 같은 방위를 확인시킨다. |
| **R3 시각 대조** | 보고 시각과 수신 시각이 1분 이상 차이 나면 지연 정보로 표시한다. |
| **R4 소리-육안 일치** | 소리 방향과 육안 방향이 다르면 '확인 요망'으로 둔다. |
| **R5 송신자 확인** | 송신자를 알 수 없는 보고는 되물어 확인한다. |
| **R6 안전 우선** | 확인 중이라도 급강하·근접 신호가 있으면 먼저 은폐를 전파한다. |

나. 제공 자료 - 오류 주입 카드 12장 [교육용가상자료] (교관이 순서를 섞어 제시)

| **카드** | **들어온 보고** | **추가로 확인되는 정보** |
| --- | --- | --- |
| **E1** | 관측1: 2시 나무 위 회전익 1기 접근 | 관측2도 2시에서 같은 기체 확인 |
| **E2** | 관측2: 9시 드론 1기 (실제는 3시) | 관측1은 3시에서 소리 청취, 무전 잡음 있음 |
| **E3** | 관측1: 12시 높은 곳 일정 속도 비행체, 소리 약함 | 일정한 직선 궤적, 깜박이는 불빛 |
| **E4** | 경계조: 10시 드론 무리 여러 기 | 날갯짓, 방향이 수시로 바뀜, 소리 없음 |
| **E5** | 관측2: 4시 회전익 1기 정지비행 | 관측1도 4시에서 윙윙 소리 확인 |
| **E6** | 관측1: 2시 회전익 1기 (3분 전 E1과 같은 내용이 늦게 도착) | 무전 지연, 시각 표기가 3분 전 |
| **E7** | 관측2: 7시 드론 2기 접근 | 관측1은 7시에서 1기만 확인, 이후 2기째 확인 |
| **E8** | 관측1: 11시 흰색 물체가 천천히 떠감 | 바람 방향과 같은 방향으로 표류, 소리 없음 |
| **E9** | 경계조: 5시 윙윙 소리, 육안 미확인 | 관측2가 5시에서 저고도 회전익 1기 육안 확인 |
| **E10** | 관측1·관측2 동시: 1시 드론 1기, 1시 드론 1기 | 같은 시각·같은 방위(중복 보고) |
| **E11** | 관측2: 8시 급강하 기체 | 관측1 확인 불가(시야 차단), 소리 급격히 커짐 |
| **E12** | 무전: '6시 드론 3기' (송신자 불명) | 관측자·경계조 누구도 송신하지 않음, 6시 방향 조용함 |

다. 작성 서식 1 - 교차검증 매트릭스

| **카드** | **적용 규칙(R번호)** | **판정
(위협/오경보/확인필요)** | **근거(관측 사실)** | **지휘자 조치** |
| --- | --- | --- | --- | --- |
| **E1** |  |  |  |  |
| **E2** |  |  |  |  |
| **E3** |  |  |  |  |
| **E4** |  |  |  |  |
| **E5** |  |  |  |  |
| **E6** |  |  |  |  |
| **E7** |  |  |  |  |
| **E8** |  |  |  |  |
| **E9** |  |  |  |  |
| **E10** |  |  |  |  |
| **E11** |  |  |  |  |
| **E12** |  |  |  |  |

| **식별 성공률** | 정답과 일치한 카드 ( ) ÷ 전체 12장 × 100 = ( )% → 목표 85% 이상 ( ) |
| --- | --- |

| **위협 놓침** | 실제 위협을 오경보로 판정한 건수 ( )건 → 목표 0건 ( ) |
| --- | --- |

라. 작성 서식 2 - 불일치 시 재확인·보고 경로(우선순위 규칙)

| **상황** | **1순위 확인** | **2순위 확인** | **상급 보고 여부·시점** |
| --- | --- | --- | --- |
| **두 관측자의 방위가 다름** |  |  |  |
| **보고가 늦게 도착(1분 이상)** |  |  |  |
| **송신자 불명 보고** |  |  |  |
| **소리만 있고 육안 미확인** |  |  |  |

마. 교관용 정답표 (훈련 후 공개)

| **카드** | **정답** | **핵심 근거** |
| --- | --- | --- |
| **E1** | 위협 | 두 관측자 방위 일치 |
| **E2** | 위협(방위 정정 3시) | 잡음, 다른 관측자 3시 청취 → 복창으로 정정 |
| **E3** | 오경보 | 높은 고도·일정 궤적·깜박이는 불빛 → 민간 항공기 추정 |
| **E4** | 오경보 | 날갯짓·방향 급변·무소음 → 조류 |
| **E5** | 위협 | 소리·육안 일치 |
| **E6** | 오경보(지연 중복) | 3분 전 E1 정보의 지연 도착 |
| **E7** | 위협(2기) | 추가 확인으로 2기 확정 |
| **E8** | 오경보 | 바람 따라 표류·무소음 → 비닐·풍선류 |
| **E9** | 위협 | 관측2 육안 확인 |
| **E10** | 위협(1기, 중복 보고) | 같은 시각·방위 → 1기로 정리 |
| **E11** | 위협(긴급) | 급강하·소리 급증 → R6 안전 우선 은폐 |
| **E12** | 확인필요→오경보 | 송신자 불명, 현장 무신호 → 되물어 확인 후 기만 가능성 보고 |

| 판정 점수 기준: '확인필요'로 두고 이후 확인을 거쳐 정답에 도달했으면 정답으로 본다. 실제 위협(E1·E2·E5·E7·E9·E10·E11)을 오경보로 버리면 성공률과 관계없이 FAIL이다. |
| --- |

| **점검 항목** | **확인** |
| --- | --- |
| 12장 모두 판정과 근거를 적었다 | ( ) |
| 근거는 관측 사실(방위·소리·궤적·시각)로 적었다 | ( ) |
| 식별 성공률과 위협 놓침 건수를 계산했다 | ( ) |
| 불일치 시 보고 경로 4가지 상황을 채웠다 | ( ) |
| 급강하·근접 신호에서는 확인보다 은폐 전파를 먼저 했다 | ( ) |

**



In [ ]:
# [미션 5 실습 코드] 교차 확인 판정 채점 및 성공률 계산
user_answers = {
    'E01': {'pred': '실제 위협'},
    'E02': {'pred': '오경보 (조류)'},
    'E03': {'pred': '오인 보고 (방위 반대)'},
    'E04': {'pred': '실제 위협'},
    'E05': {'pred': '아군/민간 승인기'},
    'E06': {'pred': '식별 혼선 (재확인 필요)'},
    'E07': {'pred': '실제 위협'},
    'E08': {'pred': '기만/통신장애'},
    'E09': {'pred': '실제 위협'},
    'E10': {'pred': '오경보 (부유물)'},
    'E11': {'pred': '실제 위협'},
    'E12': {'pred': '적 전자전 기만 시도'}
}
answer_key = {
    'E01': {'truth': '실제 위협'},
    'E02': {'truth': '오경보 (조류)'},
    'E03': {'truth': '오인 보고 (방위 반대)'},
    'E04': {'truth': '실제 위협'},
    'E05': {'truth': '아군/민간 승인기'},
    'E06': {'truth': '식별 혼선 (재확인 필요)'},
    'E07': {'truth': '실제 위협'},
    'E08': {'truth': '기만/통신장애'},
    'E09': {'truth': '실제 위협'},
    'E10': {'truth': '오경보 (부유물)'},
    'E11': {'truth': '실제 위협'},
    'E12': {'truth': '적 전자전 기만 시도'}
}
grade_mission5_cross_verification(user_answers, answer_key)
